In [ ]:
import rasterio
import xarray as xr
import glob
from pathlib import Path
import geopandas as gpd
from rasterio.mask import mask
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from scipy.ndimage import median_filter
from rasterio.io import MemoryFile
from rasterio.features import shapes
from shapely.geometry import shape
#from skimage.morphology import (
#    binary_opening, binary_closing, binary_erosion, binary_dilation,
#    disk, remove_small_objects, remove_small_holes
#)
#from scipy.ndimage import binary_fill_holes

import skimage as ski
import numpy as np

In [ ]:
years = [2016, 2017,2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026 ]
#

### 1) Define helper function for finding the data

In [ ]:
def find_files(input_folder, pattern):
    #Find files in the input folder matching the given pattern
    return sorted(glob.glob(str(input_folder / pattern)))

In [ ]:
output_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/")
output_folder.mkdir(parents=True, exist_ok=True)

s2_base = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/S2_indices")
s1_base = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence")

### 2) Combine NDSI & Ratio files and create composite from east and west

In [ ]:
def create_composite(ndsi_east_file, ndsi_west_file,
                     ratio_east_file, ratio_west_file,
                     output_file):

    # Merge east and west NDSI rasters
    with rasterio.open(ndsi_east_file) as east, \
         rasterio.open(ndsi_west_file) as west:

        ndsi, ndsi_transform = merge([east, west], method="max")
        meta = east.meta.copy()

    # Merge east and west ratio rasters
    with rasterio.open(ratio_east_file) as east, \
         rasterio.open(ratio_west_file) as west:

        ratio, ratio_transform = merge([east, west], method="max")

    # Combine NDSI and ratio using logical OR
    glacier_extent = (
        (ndsi[0] == 1) | (ratio[0] == 1)
    ).astype(np.uint8)

    # Save he final output
    meta.update({
        "height": glacier_extent.shape[0],
        "width": glacier_extent.shape[1],
        "transform": ndsi_transform,
        "count": 1,
        "dtype": "uint8",
        "nodata": 255,
    })

    with rasterio.open(output_file, "w", **meta) as dst:
        dst.write(glacier_extent, 1)

    print(f"Saved final glacier extent: {output_file}")

In [ ]:
for year in years:

    print(f"\n===== {year} =====")

    input_folder_s2 = s2_base / str(year)

    # Find the four input rasters
    ndsi_east = find_files(
        input_folder_s2, "S2_mosaic_east_*_ndsi_*.tiff"
    )
    ndsi_west = find_files(
        input_folder_s2, "S2_mosaic_west_*_ndsi_*.tiff"
    )
    ratio_east = find_files(
        input_folder_s2, "S2_mosaic_east_*_ratio_10.tiff"
    )
    ratio_west = find_files(
        input_folder_s2, "S2_mosaic_west_*_ratio_10.tiff"
    )

    if not all([
        len(ndsi_east) == 1,
        len(ndsi_west) == 1,
        len(ratio_east) == 1,
        len(ratio_west) == 1,
    ]):
        print(f"Skipping {year}: expected exactly four input files.")
        continue

    # Final output only
    output_file = input_folder_s2 / f"glacier_composite_{year}_OR.tiff"

    create_composite(
        ndsi_east[0],
        ndsi_west[0],
        ratio_east[0],
        ratio_west[0],
        output_file
    )

### 3) Mask ascending and descending mosaics with threshold 

In [ ]:
thr = 0.4

In [ ]:
def apply_threshold(input_filename, output_filename, thr=thr):
    with rasterio.open(input_filename) as src:
        data = src.read(1).astype(np.float32)
        meta = src.meta.copy()
        nodata = src.nodata

    # invalid = NaN or nodata value
    invalid = np.isnan(data)
    if nodata is not None and not np.isnan(nodata):
        invalid |= (data == nodata)

    # 1 = coherence below threshold, 0 = above, 255 = nodata
    glacier = np.where(data < thr, 1, 0).astype("uint8")
    glacier[invalid] = 255

    meta.update(count=1, dtype="uint8", nodata=255)

    with rasterio.open(output_filename, "w", **meta) as dst:
        dst.write(glacier, 1)

    print(f"Saved: {output_filename}")


for year in years:
    print(f"\n===== {year} =====")
    input_folder_s2 = s2_base / str(year)
    input_folder_s1 = s1_base / str(year)

    # find ascending / descending coherence files (note the f-strings)
    s1_coherence_files_asc = find_files(input_folder_s1, f"coh_mosaic_ascending_{year}.tiff")
    s1_coherence_files_desc = find_files(input_folder_s1, f"coh_mosaic_descending_{year}.tiff")
    print(f"Found {len(s1_coherence_files_asc)} S1 coherence files (ascending).")
    print(f"Found {len(s1_coherence_files_desc)} S1 coherence files (descending).")


    # apply threshold to S1 coherence files (pass file paths, not arrays)
    apply_threshold(
        s1_coherence_files_asc[0],
        input_folder_s1 / f"ascending_{year}_thresh_{thr}.tiff",
    )
    apply_threshold(
        s1_coherence_files_desc[0],
        input_folder_s1 / f"descending_{year}_thresh_{thr}.tiff",
    )

### 4) Align all rasters spatially

In [ ]:
def align_to_reference(src_array, src_transform, src_crs, ref_meta,
                       src_nodata=None, dst_nodata=None,
                       resampling=Resampling.nearest):
    fill = dst_nodata if dst_nodata is not None else 0
    aligned = np.full((ref_meta["height"], ref_meta["width"]), fill, dtype=src_array.dtype)
    reproject(
        source=src_array,
        destination=aligned,
        src_transform=src_transform,
        src_crs=src_crs,
        src_nodata=src_nodata,
        dst_transform=ref_meta["transform"],
        dst_crs=ref_meta["crs"],
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    return aligned

In [ ]:
for year in years:
    print(f"\n===== {year} =====")
    input_folder_s2 = s2_base / str(year)
    input_folder_s1 = s1_base / str(year)

    s2_file   = input_folder_s2 / f"glacier_composite_{year}.tiff"
    asc_file  = input_folder_s1 / f"ascending_{year}_thresh_{thr}.tiff"
    desc_file = input_folder_s1 / f"descending_{year}_thresh_{thr}.tiff"

    missing = [p.name for p in (s2_file, asc_file, desc_file) if not p.exists()]
    if missing:
        print(f"Missing for {year}: {missing}, skipping.")
        continue

    # load everything from disk
    with rasterio.open(asc_file) as src:
        s1_asc = src.read(1)
        s1_asc_meta = src.meta.copy()
    with rasterio.open(desc_file) as src:
        s1_desc = src.read(1)
        s1_desc_meta = src.meta.copy()
    with rasterio.open(s2_file) as src:
        s2 = src.read(1)
        s2_transform = src.transform
        s2_crs = src.crs

    ref_meta = s1_asc_meta  # reference grid

    s2_aligned = align_to_reference(
        s2, s2_transform, s2_crs, ref_meta, src_nodata=None, dst_nodata=0,
    )
    s1_asc_aligned = align_to_reference(
        s1_asc, s1_asc_meta["transform"], s1_asc_meta["crs"], ref_meta,
        src_nodata=255, dst_nodata=255,
    )
    s1_desc_aligned = align_to_reference(
        s1_desc, s1_desc_meta["transform"], s1_desc_meta["crs"], ref_meta,
        src_nodata=255, dst_nodata=255,
    )

    combined = (
        (s2_aligned == 1) | (s1_asc_aligned == 1) | (s1_desc_aligned == 1)
    ).astype(np.uint8)

    combined_meta = ref_meta.copy()
    combined_meta.update({"count": 1, "dtype": "uint8", "nodata": None})

    combined_folder = output_folder / "S1_S2" / str(year)
    combined_folder.mkdir(parents=True, exist_ok=True)
    combined_filename = combined_folder / f"s1_s2_asc_desc_combined_{year}_{thr}.tiff"

    with rasterio.open(combined_filename, "w", **combined_meta) as dest:
        dest.write(combined, 1)
    print("Wrote:", combined_filename)

    s2_bin = (s2_aligned == 1).astype(np.uint8)
    s1_bin = ((s1_asc_aligned == 1) | (s1_desc_aligned == 1)).astype(np.uint8)

    layer_folders = {
        "s2": output_folder / "S2" / str(year),
        "s1": output_folder / "S1" / str(year),
    }

    for name, arr in {"s2": s2_bin, "s1": s1_bin}.items():
        layer_folders[name].mkdir(parents=True, exist_ok=True)
        out_path = layer_folders[name] / f"{name}_aligned_{year}_{thr}.tiff"
        with rasterio.open(out_path, "w", **combined_meta) as dst:
            dst.write(arr, 1)
        print("Wrote:", out_path)

### 6) Clip to glacier outlines from 1997

In [ ]:
mask_filename = "/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/glacier_outline/glacier_inventory/SGIhom_1997_v2025.shp"
glacier_mask = gpd.read_file(mask_filename)

def _write_tif(filename, array, crs, transform):
    out_meta = {
        "driver": "GTiff",
        "height": array.shape[0],
        "width": array.shape[1],
        "count": 1,
        "dtype": "uint8",
        "crs": crs,
        "transform": transform,
        "nodata": 0,
        "compress": "lzw",
    }
    with rasterio.open(filename, "w", **out_meta) as dst:
        dst.write(array.astype(np.uint8), 1)
    print(f"Saved: {filename}")

def process_binary_mask(binary_mask, transform, crs, glacier_mask,
                        disk_size=5, output_filename=None,
                        glacier_output_filename=None,
                        masked_input_filename=None):
    """Mask a binary raster with glacier polygons, apply morphological filtering and save the result and the glacier mask before the filtering."""
    binary_mask = binary_mask.astype(np.uint8)

    meta = {
        "driver": "GTiff",
        "height": binary_mask.shape[0],
        "width": binary_mask.shape[1],
        "count": 1,
        "dtype": "uint8",
        "crs": crs,
        "transform": transform,
    }

    glacier_mask_reprojected = glacier_mask.to_crs(crs)

    with MemoryFile() as memfile:
        with memfile.open(**meta) as tmp:
            tmp.write(binary_mask, 1)

        with memfile.open() as src:
            masked_array, masked_transform = mask(
                src,
                glacier_mask_reprojected.geometry,
                crop=True,
                filled=False,
            )

    # True inside the glacier polygons
    glacier_area = ~np.ma.getmaskarray(masked_array[0])

    # input clipped to glacier 
    binary = masked_array[0].filled(0).astype(np.uint8)
    
    if glacier_output_filename is not None:
        _write_tif(glacier_output_filename, binary, crs, masked_transform)

    struct = ski.morphology.disk(disk_size)
    opened = ski.morphology.opening(binary, struct).astype(np.uint8)

    #closed = ski.morphology.closing(opened, struct)

    #closed[~glacier_area] = 0

    if output_filename is not None:
        _write_tif(output_filename, opened, crs, masked_transform)

    return opened, masked_transform, glacier_area

In [ ]:
disk_sizes = [3]

for year in years:
    print(f"\n===== {year} =====")
    print(output_folder)

    inputs = {
        "s2": output_folder / "S2" / str(year) /
              f"s2_aligned_{year}_{thr}.tiff",

        "s1": output_folder / "S1" / str(year) /
              f"s1_aligned_{year}_{thr}.tiff",

        "combined": output_folder / "S1_S2" / str(year) /
                    f"s1_s2_asc_desc_combined_{year}_{thr}.tiff",
    }

    # Check that all inputs exist
    missing = [p.name for p in inputs.values() if not p.exists()]

    if missing:
        print(f"Missing for {year}: {missing}, skipping.")
        continue

    # Process each input raster
    for name, path in inputs.items():

        for i in disk_sizes:

            # Generate output filenames for this disk size
            outputs = {
                "s2": output_folder / "S2" / str(year) /
                      f"s2_glacier_{year}_{thr}_{i}_openend.tiff",

                "s1": output_folder / "S1" / str(year) /
                      f"s1_glacier_{year}_{thr}_{i}_openend.tiff",

                "combined": output_folder / "S1_S2" / str(year) /
                            f"s1_s2_asc_desc_combined_glacier_{year}_{thr}_{i}_openend.tiff",
            }

            glacier_outputs = {
                "s2": output_folder / "S2" / str(year) /
                      f"glacier_mask_s2_{thr}_{i}.tiff",

                "s1": output_folder / "S1" / str(year) /
                      f"glacier_mask_s1_{thr}_{i}.tiff",

                "combined": output_folder / "S1_S2" / str(year) /
                            f"glacier_mask_s1_s2_{thr}_{i}.tiff",
            }

            # Read input raster
            with rasterio.open(path) as src:
                arr = src.read(1)
                transform = src.transform
                crs = src.crs

            # Process the raster
            process_binary_mask(
                arr,
                transform,
                crs,
                glacier_mask,
                disk_size=i,
                output_filename=outputs[name],
                #glacier_output_filename=glacier_outputs[name]
            )

            print(f"Processed {name.upper()} for {year}, disk size {i}")

### 8) Convert into polygons

In [ ]:
def raster_to_polygons(binary_raster, transform, crs):
    """Convert pixels == 1 of a binary raster into polygons."""
    binary_raster = binary_raster.astype(np.uint8)
    mask = binary_raster == 1  

    geoms = [
        shape(geom)
        for geom, value in shapes(binary_raster, mask=mask, transform=transform)
    ]
    return gpd.GeoDataFrame({"value": [1] * len(geoms)}, geometry=geoms, crs=crs)


version = "v0.4"

layers = {
    # name: (input glacier raster, output shapefile)
    "s2": (
        lambda y, d: output_folder / "S2" / str(y) / f"s2_glacier_{y}_{thr}_{d}_openend.tiff",
        lambda y, d: output_folder / "S2" / str(y) / f"s2_glacier_outlines_{y}_{version}_{d}.shp",
    ),
    "s1": (
        lambda y, d: output_folder / "S1" / str(y) / f"s1_glacier_{y}_{thr}_{d}_openend.tiff",
        lambda y, d: output_folder / "S1" / str(y) / f"s1_glacier_outlines_{y}_{version}_{d}.shp",
    ),
    "combined": (
        lambda y, d: output_folder / "S1_S2" / str(y) / f"s1_s2_asc_desc_combined_glacier_{y}_{thr}_{d}_openend.tiff",
        lambda y, d: output_folder / "S1_S2" / str(y) / f"s1_s2_glacier_outlines_{y}_{version}_{d}.shp",
    ),
}

In [ ]:
for year in years:
    for d in disk_sizes:
        print(f"\n===== {year} | disk size {d} =====")

        in_paths = {name: build_in(year, d) for name, (build_in, _) in layers.items()}
        missing = [p.name for p in in_paths.values() if not p.exists()]
        if missing:
            print(f"Missing for {year}, disk {d}: {missing}, skipping.")
            continue

        for name, (_, build_out) in layers.items():
            with rasterio.open(in_paths[name]) as src:
                arr = src.read(1)
                transform = src.transform
                crs = src.crs

            gdf = raster_to_polygons(arr, transform, crs)

            out_path = build_out(year, d)
            out_path.parent.mkdir(parents=True, exist_ok=True)
            gdf.to_file(out_path)

            print(f"{name}: {len(gdf)} polygons -> {out_path}")

### 9) Extract area in km2

In [ ]:
from shapely.geometry import box
import pandas as pd

rows = []

for year in years:
    print(f"\n===== {year} =====")

    input_folder_s2 = s2_base / str(year)

    # Find east and west NDSI mosaics
    ndsi_files = {
        "east": find_files(input_folder_s2, "S2_mosaic_east_*_ndsi_*.tiff"),
        "west": find_files(input_folder_s2, "S2_mosaic_west_*_ndsi_*.tiff"),
    }

    # Read raster extents once per year (not per layer / disk size)
    region_extents = {}  # region -> (extent polygon, crs)
    for region, files in ndsi_files.items():
        if len(files) != 1:
            print(
                f"Expected one {region} NDSI mosaic for {year}, "
                f"found {len(files)}. Skipping region."
            )
            continue

        with rasterio.open(files[0]) as src:
            b = src.bounds
            region_extents[region] = gpd.GeoDataFrame(
                geometry=[box(b.left, b.bottom, b.right, b.top)],
                crs=src.crs,
            )

    for d in disk_sizes:
        for name, (_, build_out) in layers.items():
            path = build_out(year, d)

            if not path.exists():
                print(f"Missing: {path.name}, skipping.")
                continue

            gdf = gpd.read_file(path)

            if gdf.crs is None or not gdf.crs.is_projected:
                raise ValueError(
                    f"{path.name}: CRS is not projected; "
                    "area in m² would be wrong."
                )

            for region, extent in region_extents.items():
                # Reproject extent to glacier polygon CRS
                extent_proj = extent.to_crs(gdf.crs)

                # Clip glacier polygons to east/west extent
                clipped = gpd.clip(gdf, extent_proj)

                area_km2 = clipped.geometry.area.sum() / 1e6

                rows.append({
                    "year": year,
                    "disk_size": d,
                    "layer": name,
                    "region": region,
                    "n_polygons": len(clipped),
                    "area_km2": round(area_km2, 3),
                })

                print(
                    f"{year} disk={d:<3} {name:<9} {region:<5} "
                    f"{area_km2:8.2f} km² ({len(clipped)} polygons)"
                )

results = pd.DataFrame(rows)

wide = results.pivot_table(
    index=["layer", "region"],
    columns="year",
    values="area_km2",
).reset_index()

wide.to_csv(output_folder / f"glacier_area_wide_{version}.csv", index=False)

### Try out another method

In [ ]:
import rasterio
import numpy as np
from skimage.morphology import remove_small_objects, remove_small_holes

mask_filename = "/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/glacier_outline/glacier_inventory/SGIhom_1997_v2025.shp"
glacier_mask = gpd.read_file(mask_filename)

def _write_tif(filename, array, crs, transform):
    out_meta = {
        "driver": "GTiff",
        "height": array.shape[0],
        "width": array.shape[1],
        "count": 1,
        "dtype": "uint8",
        "crs": crs,
        "transform": transform,
        "nodata": 0,
        "compress": "lzw",
    }
    with rasterio.open(filename, "w", **out_meta) as dst:
        dst.write(array.astype(np.uint8), 1)
    print(f"Saved: {filename}")


def process_binary_mask(binary_mask, transform, crs, glacier_mask,
                        min_patch_area=10_000,   # m², remove isolated patches smaller than this
                        max_hole_area=10_000,    # m², fill holes smaller than this
                        output_filename=None,
                        glacier_output_filename=None):
    """Mask a binary raster with glacier polygons, remove small patches,
    fill small holes, and save the result and the clipped input."""
    binary_mask = binary_mask.astype(np.uint8)

    meta = {
        "driver": "GTiff",
        "height": binary_mask.shape[0],
        "width": binary_mask.shape[1],
        "count": 1,
        "dtype": "uint8",
        "crs": crs,
        "transform": transform,
    }

    glacier_mask_reprojected = glacier_mask.to_crs(crs)

    with MemoryFile() as memfile:
        with memfile.open(**meta) as tmp:
            tmp.write(binary_mask, 1)

        with memfile.open() as src:
            masked_array, masked_transform = mask(
                src,
                glacier_mask_reprojected.geometry,
                crop=True,
                filled=False,
            )

    # True inside the glacier polygons
    glacier_area = ~np.ma.getmaskarray(masked_array[0])

    # input clipped to glacier
    binary = masked_array[0].filled(0).astype(np.uint8)
    #if glacier_output_filename is not None:
    #    _write_tif(glacier_output_filename, binary, crs, masked_transform)

    # thresholds: m² -> pixels
    px_area = abs(masked_transform.a * masked_transform.e)
    min_px = max(1, int(round(min_patch_area / px_area)))
    hole_px = max(1, int(round(max_hole_area / px_area)))

    # 1. remove small isolated patches
    cleaned = ski.morphology.remove_small_objects(
        binary.astype(bool), max_size=min_px - 1, connectivity=2
    )

    # 2. Fill small wholes
    cleaned = ski.morphology.remove_small_holes(
        cleaned, max_size=hole_px - 1, connectivity=2
    )  
    
    cleaned = cleaned.astype(np.uint8)

    # keep result strictly inside the glacier polygons
    cleaned[~glacier_area] = 0

    if output_filename is not None:
        _write_tif(output_filename, cleaned, crs, masked_transform)

    return cleaned, masked_transform, glacier_area

In [ ]:
for year in years:
    print(f"\n===== {year} =====")
    print(output_folder)

    inputs = {
        "s2": output_folder / "S2" / str(year) /
              f"s2_aligned_{year}_{thr}.tiff",

        "s1": output_folder / "S1" / str(year) /
              f"s1_aligned_{year}_{thr}.tiff",

        "combined": output_folder / "S1_S2" / str(year) /
                    f"s1_s2_asc_desc_combined_{year}_{thr}.tiff",
    }

    # Check that all inputs exist
    missing = [p.name for p in inputs.values() if not p.exists()]

    if missing:
        print(f"Missing for {year}: {missing}, skipping.")
        continue

    # Process each input raster
    for name, path in inputs.items():

            # Generate output filenames for this disk size
            outputs = {
                "s2": output_folder / "S2" / str(year) /
                      f"s2_glacier_{year}_{thr}_minPatch.tiff",

                "s1": output_folder / "S1" / str(year) /
                      f"s1_glacier_{year}_{thr}_minPatch.tiff",

                "combined": output_folder / "S1_S2" / str(year) /
                            f"s1_s2_asc_desc_combined_glacier_{year}_{thr}_minPatch.tiff",
            }

            glacier_outputs = {
                "s2": output_folder / "S2" / str(year) /
                      f"glacier_mask_s2_{thr}_20000.tiff",

                "s1": output_folder / "S1" / str(year) /
                      f"glacier_mask_s1_{thr}_20000.tiff",

                "combined": output_folder / "S1_S2" / str(year) /
                            f"glacier_mask_s1_s2_{thr}_20000.tiff",
            }

            # Read input raster
            with rasterio.open(path) as src:
                arr = src.read(1)
                transform = src.transform
                crs = src.crs

            # Process the raster
            process_binary_mask(
                arr,
                transform,
                crs,
                glacier_mask,
                min_patch_area= 10000,
                max_hole_area= 20000,
                output_filename=outputs[name],
                #glacier_output_filename=glacier_outputs[name]
            )

In [ ]:
def raster_to_polygons(binary_raster, transform, crs):
    """Convert pixels == 1 of a binary raster into polygons."""
    binary_raster = binary_raster.astype(np.uint8)
    mask = binary_raster == 1  

    geoms = [
        shape(geom)
        for geom, value in shapes(binary_raster, mask=mask, transform=transform)
    ]
    return gpd.GeoDataFrame({"value": [1] * len(geoms)}, geometry=geoms, crs=crs)


version = "v0.4"

layers = {
    # name: (input glacier raster, output shapefile)
    # "s2": (
    #     lambda y, d: output_folder / "S2" / str(y) / f"s2_glacier_{y}_{thr}_minPatch.tiff",
    #     lambda y, d: output_folder / "S2" / str(y) / f"s2_glacier_outlines_{y}_{version}_minPatch.shp",
    # ),
    # "s1": (
    #     lambda y, d: output_folder / "S1" / str(y) / f"s1_glacier_{y}_{thr}_{d}_closed.tiff",
    #     lambda y, d: output_folder / "S1" / str(y) / f"s1_glacier_outlines_{y}_{version}_minPatch.shp",
    # ),
    "combined": (
        lambda y: output_folder / "S1_S2" / str(y) / f"s1_s2_asc_desc_combined_{y}_{thr}.tiff",
        lambda y: output_folder / "S1_S2" / str(y) / f"s1_s2_glacier_outlines_{y}_{version}_3.shp",
    ),
}

In [ ]:
for year in years:
    print(f"\n===== {year} =====")

    in_paths = {name: build_in(year) for name, (build_in, _) in layers.items()}
    missing = [p.name for p in in_paths.values() if not p.exists()]
    if missing:
        print(f"Missing for {year}: {missing}, skipping.")
        continue

    for name, (_, build_out) in layers.items():
        with rasterio.open(in_paths[name]) as src:
            arr = src.read(1)
            transform = src.transform
            crs = src.crs

        gdf = raster_to_polygons(arr, transform, crs)

        out_path = build_out(year)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        gdf.to_file(out_path)

        print(f"{name}: {len(gdf)} polygons -> {out_path}")

In [ ]:
from shapely.geometry import box
import pandas as pd

rows = []

for year in years:
    print(f"\n===== {year} =====")

    input_folder_s2 = s2_base / str(year)

    # Find east and west NDSI mosaics
    ndsi_files = {
        "east": find_files(input_folder_s2, "S2_mosaic_east_*_ndsi_*.tiff"),
        "west": find_files(input_folder_s2, "S2_mosaic_west_*_ndsi_*.tiff"),
    }

    # Read raster extents once per year
    region_extents = {}
    for region, files in ndsi_files.items():
        if len(files) != 1:
            print(
                f"Expected one {region} NDSI mosaic for {year}, "
                f"found {len(files)}. Skipping region."
            )
            continue

        with rasterio.open(files[0]) as src:
            b = src.bounds
            region_extents[region] = gpd.GeoDataFrame(
                geometry=[box(b.left, b.bottom, b.right, b.top)],
                crs=src.crs,
            )

    for name, (_, build_out) in layers.items():
        path = build_out(year)

        if not path.exists():
            print(f"Missing: {path.name}, skipping.")
            continue

        gdf = gpd.read_file(path)

        if gdf.crs is None or not gdf.crs.is_projected:
            raise ValueError(
                f"{path.name}: CRS is not projected; area in m² would be wrong."
            )

        for region, extent in region_extents.items():
            extent_proj = extent.to_crs(gdf.crs)
            clipped = gpd.clip(gdf, extent_proj)
            area_km2 = clipped.geometry.area.sum() / 1e6

            rows.append({
                "year": year,
                "layer": name,
                "region": region,
                "n_polygons": len(clipped),
                "area_km2": round(area_km2, 3),
            })

            print(
                f"{year} {name:<9} {region:<5} "
                f"{area_km2:8.2f} km² ({len(clipped)} polygons)"
            )
